# 🧠 Implementing Q-Learning (DQN) in Keras


## 📋 Overview

In every notebook so far in this module I've trained a network to map a fixed input to a fixed target — an image to a label, a noisy image to a clean one, a latent vector to a fake sample. Reinforcement learning flips that setup: there's no fixed dataset at all. Instead there's an **agent** interacting with an **environment**, taking actions, and only finding out afterward — via a reward — whether that was a good idea.

Here I build a **Deep Q-Network (DQN)**: a small Keras model that learns to balance a pole on a cart (the classic CartPole environment from Gymnasium) purely from trial and error. No labeled "correct action" ever exists — the network has to estimate, for a given state, how good each possible action is, and get better at that estimate over thousands of attempts.

This connects directly to [[reinforcement_learning]], which I first met back in course 01 in its classical (non-neural) form. Here the same agent/environment/reward loop gets a neural network doing the value estimation instead of a lookup table — which is the whole reason it's called *deep* Q-learning.

**What I'll build:**
1. A Gymnasium `CartPole-v1` environment
2. A small Q-network that scores each action given a state
3. An experience replay buffer + epsilon-greedy exploration
4. A training loop and a greedy evaluation loop
5. Three practice extensions: a different network architecture, adaptive exploration, and a custom reward function


## 🧩 Theory

### The reinforcement learning loop

At every timestep $t$, the agent observes a **state** $s_t$, picks an **action** $a_t$, and the environment returns a **reward** $r_t$ and a **next state** $s_{t+1}$. The goal is to learn a policy that maximizes *cumulative* future reward, not just the immediate one.

### Q-values and the Bellman equation

A **Q-value** $Q(s,a)$ estimates the total future reward of taking action $a$ in state $s$, then acting optimally afterward. The classic (tabular) Q-learning update rule is:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\Big]$$

Where $\alpha$ is the learning rate, $\gamma$ is the **discount factor** (how much future reward matters relative to immediate reward), and $\max_{a'}Q(s',a')$ is the best Q-value achievable from the next state.

With a neural network standing in for the Q-table, the update becomes a regression problem: for the action actually taken, the network is trained to move its prediction toward a **target**:

$$\text{target} = r + \gamma \max_{a'} Q_\theta(s', a')$$

$$\mathcal{L}(\theta) = \Big(Q_\theta(s,a) - \text{target}\Big)^2$$

only the Q-value of the action that was actually taken gets updated per step — the other action's predicted Q-value is left untouched by copying the network's own current prediction into the target vector before computing the loss.

**⚠️ A simplification worth flagging:** a "real" DQN (the 2015 DeepMind paper) uses **two** networks — an online network being trained, and a slowly-updated **target network** that supplies the $\max_{a'}Q(s',a')$ term. This lab uses a single network for both roles. That's simpler to implement, but it means the target the network is chasing shifts every time the network itself updates — a moving target that can make training noisier and less stable. **Telecom/RF 📡 analogy:** it's like tuning an adaptive filter using a reference signal that's *also* being adjusted by the same filter in real time — instead of holding the reference steady for a while before updating it. Workable, but prone to oscillation compared to a stable reference.

### Epsilon-greedy exploration

The agent has to balance **exploring** (trying actions to learn their value) against **exploiting** (using the best action it already knows about):

$$a_t = \begin{cases} \text{random action}, & \text{with probability } \epsilon \\ \arg\max_a Q_\theta(s_t, a), & \text{with probability } 1-\epsilon \end{cases}$$

$\epsilon$ starts near 1 (mostly random) and decays over episodes:

$$\epsilon \leftarrow \max(\epsilon_{\min},\ \epsilon \cdot \epsilon_{\text{decay}})$$

**Telecom/RF 📡 analogy:** this is exactly the explore/exploit tradeoff in **cognitive radio spectrum sensing** — a radio can keep scanning other channels looking for a better one (explore) or stay locked onto the channel it already knows is decent (exploit). Early on, when it knows little about the spectrum, it scans aggressively; as it builds confidence in a good channel, it scans less and stays locked more — precisely the shape of epsilon decay.

### Experience replay

Instead of training on transitions in the order they happen (which are highly correlated — consecutive states barely differ), transitions $(s, a, r, s', \text{done})$ are stored in a buffer and **sampled randomly** in minibatches for training.

**Telecom/RF 📡 analogy:** this is the same reason a communications system uses an **interleaver** before forward error correction — burst errors are correlated in time, so data gets shuffled before decoding to spread that correlation out. Experience replay shuffles *training samples* for the same underlying reason: breaking temporal correlation makes the learning signal much better behaved.

### Reward shaping

The reward function doesn't have to be the environment's default. Shaping the reward — giving denser, more informative feedback instead of a single sparse terminal signal — is like replacing a pass/fail per-packet check with a continuous quality metric such as EVM: the smoother the feedback, the more useful gradient the learner actually gets from every single step, not just at the end.


## Part 1 — 🌍 Setting Up the Environment

[Gymnasium](https://gymnasium.farama.org/) is the maintained successor to OpenAI Gym — it provides ready-made RL environments, including `CartPole-v1`: a cart on a frictionless track with a pole hinged on top. The agent pushes the cart left or right; the episode ends if the pole tips past ~12° or the cart drifts too far off-center. The reward is +1 for every timestep the pole stays upright.

The state is a 4-value vector: cart position, cart velocity, pole angle, and pole angular velocity. The action space is discrete: 0 (push left) or 1 (push right).


In [ ]:
# Install/pin compatible versions — Gymnasium + TensorFlow + NumPy have had
# version-compatibility friction in hosted notebook environments, so this
# pins a known-working combination before importing anything.
!pip install --upgrade numpy==1.26.4 --quiet
!pip install gymnasium --quiet


In [ ]:
import numpy as np
import random
from collections import deque

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

import gymnasium as gym

# Reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)


In [ ]:
env = gym.make('CartPole-v1')

# Cast explicitly to Python int -- Gymnasium's Discrete.n (and some
# observation_space.shape entries) can come back as numpy.int64, which
# Keras 3's Dense layer rejects for `units` even though the value itself
# (e.g. 2) is perfectly valid.
state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

print(f"State size: {state_size}")
print(f"Action size: {action_size}")


## Part 2 — 🏗️ Defining the Q-Network

The network takes a 4-value state as input and outputs one Q-value per action (2 outputs, one per discrete action). It's deliberately small — two hidden layers of 24 units — since CartPole's state space is low-dimensional.

The output layer uses a **linear** activation, not softmax — Q-values are unbounded real-valued estimates of future reward, not a probability distribution. The loss is mean squared error, since this is a regression problem (predict a Q-value), not classification.


In [ ]:
def build_model(state_size, action_size):
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(24, activation='relu'),
        Dense(24, activation='relu'),
        Dense(action_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model = build_model(state_size, action_size)
model.summary()


## Part 3 — 🔄 Replay Memory, Epsilon-Greedy Action Selection, and the Training Loop

Three pieces work together here:

| Piece | Role |
|---|---|
| `memory` (deque) | Fixed-size buffer of past transitions, used for experience replay |
| `remember()` | Appends a transition `(state, action, reward, next_state, done)` to memory |
| `act()` | Epsilon-greedy action selection — random with probability $\epsilon$, else $\arg\max_a Q(s,a)$ |
| `replay()` | Samples a random minibatch from memory, computes Bellman targets, and trains the network on them |


In [ ]:
gamma = 0.95        # discount factor
epsilon = 1.0        # initial exploration rate
epsilon_min = 0.01   # exploration floor
epsilon_decay = 0.995
memory = deque(maxlen=2000)

def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def act(state):
    global epsilon
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state, verbose=0)
    return np.argmax(q_values[0])


In [ ]:
def replay(batch_size):
    global epsilon
    if len(memory) < batch_size:
        return

    minibatch = random.sample(memory, batch_size)

    states = np.array([t[0][0] for t in minibatch])
    actions = np.array([t[1] for t in minibatch])
    rewards = np.array([t[2] for t in minibatch])
    next_states = np.array([t[3][0] for t in minibatch])
    dones = np.array([t[4] for t in minibatch])

    # Current Q-value predictions for the whole batch — this becomes the
    # target vector, except for the action actually taken at each step.
    q_values = model.predict(states, verbose=0)
    q_values_next = model.predict(next_states, verbose=0)

    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += gamma * np.amax(q_values_next[i])
        q_values[i][actions[i]] = target

    model.fit(states, q_values, epochs=1, verbose=0)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay


### Training loop

Each episode resets the environment, then steps through the environment up to `max_timesteps`, choosing actions epsilon-greedily and remembering every transition. Note the **reward shaping** already present here: the environment's own reward is +1 per timestep, but this loop overrides it to **-10 whenever the episode ends early** (`done=True` before hitting the timestep cap) — a stronger penalty signal for failure than the environment provides by default, on top of whatever positive reward accumulated from surviving.

Training via `replay()` happens periodically (`train_frequency`), not on every single step, which reduces how often the network is retrained per episode.


In [ ]:
episodes = 10
max_timesteps = 200
batch_size = 64
train_frequency = 5

for e in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time in range(max_timesteps):
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])

        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

        if time % train_frequency == 0:
            replay(batch_size)

env.close()


## Part 4 — 📊 Evaluating the Trained Agent

Evaluation drops exploration entirely — the agent always picks $\arg\max_a Q(s,a)$ greedily, since the goal now is to measure how good the learned policy actually is, not to keep learning.

**⚠️ A practical note:** `env.render()` requires the environment to have been created with a `render_mode` (e.g. `gym.make('CartPole-v1', render_mode='human')`). As written below, the environment is created without one, so calling `.render()` will either no-op or raise a warning depending on the Gymnasium version — worth setting `render_mode` explicitly if visual playback is the goal. I've kept the code exactly as the source lab wrote it.


In [ ]:
eval_env = gym.make('CartPole-v1')
eval_episodes = 10
eval_max_timesteps = 500

for e in range(eval_episodes):
    state, _ = eval_env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time in range(eval_max_timesteps):
        eval_env.render()
        q_values = model.predict(state, verbose=0)
        action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        state = np.reshape(next_state, [1, state_size])
        total_reward += reward

        if done:
            print(f"Evaluation Episode: {e+1}/{eval_episodes}, Score: {time}")
            break

eval_env.close()


## 🎯 Practice 1 — A Different Network Architecture

Does a wider network (32 units instead of 24, still 2 hidden layers) learn a better policy? Here's a variant to compare against the original.

**⚠️ Flagging a real inconsistency:** the source lab's solution for this exercise redefines `act()` to `return env.action_space.sample()` — i.e. it **always returns a fully random action**, completely ignoring the trained model's Q-value predictions. That silently defeats the entire point of the exercise: it can't actually compare "24-unit network" vs. "32-unit network" performance, because with this `act()`, *neither* network's predictions are ever used to choose an action during training or evaluation. I've corrected this below to keep the original epsilon-greedy `act()` logic, since a random-action-only agent isn't a meaningful architecture comparison. The rebuilt network itself is included as originally written. The source also imports `from keras.models import Sequential` (bare `keras`) here, inconsistent with the rest of the notebook's `tensorflow.keras` imports — both APIs are compatible in this TF version, but I've kept the `tensorflow.keras` import style used everywhere else for consistency.


In [ ]:
def build_model_wide(state_size, action_size):
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(32, activation='relu'),
        Dense(32, activation='relu'),
        Dense(action_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model_wide = build_model_wide(state_size, action_size)
model_wide.summary()


In [ ]:
# Fresh exploration/memory state so this comparison isn't contaminated
# by the epsilon decay and replay buffer from the earlier training run.
epsilon = 1.0
memory_wide = deque(maxlen=2000)

def remember_wide(state, action, reward, next_state, done):
    memory_wide.append((state, action, reward, next_state, done))

def act_wide(state):
    # Kept as genuine epsilon-greedy action selection — NOT the source lab's
    # env.action_space.sample()-only stub, which would make this comparison
    # meaningless (see note above).
    global epsilon
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model_wide.predict(state, verbose=0)
    return np.argmax(q_values[0])

def replay_wide(batch_size):
    global epsilon
    if len(memory_wide) < batch_size:
        return

    minibatch = random.sample(memory_wide, batch_size)
    states = np.array([t[0][0] for t in minibatch])
    actions = np.array([t[1] for t in minibatch])
    rewards = np.array([t[2] for t in minibatch])
    next_states = np.array([t[3][0] for t in minibatch])
    dones = np.array([t[4] for t in minibatch])

    q_values = model_wide.predict(states, verbose=0)
    q_values_next = model_wide.predict(next_states, verbose=0)

    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += gamma * np.amax(q_values_next[i])
        q_values[i][actions[i]] = target

    model_wide.fit(states, q_values, epochs=1, verbose=0)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

env_wide = gym.make('CartPole-v1')

for e in range(episodes):
    state, _ = env_wide.reset()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        action = act_wide(state)
        next_state, reward, terminated, truncated, _ = env_wide.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])

        remember_wide(state, action, reward, next_state, done)
        state = next_state

        if done:
            print(f"[Wide net] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

        if time % train_frequency == 0:
            replay_wide(batch_size)

env_wide.close()


## ⚙️ Practice 2 — Adaptive Exploration Rate

Instead of decaying $\epsilon$ by a fixed multiplier every replay call regardless of how training is going, this makes the decay **performance-aware**: if the agent recently scored above a threshold (close to "solving" the episode), epsilon drops faster; if it's still struggling, epsilon decays at the normal, slower rate.

**⚠️ Minor consistency note:** the source lab's solution for this exercise unpacks `env.step()` as `next_state, reward, done, truncated, _ = env.step(action)` — binding the variable name `done` directly to what Gymnasium actually returns as `terminated`. That's a naming mismatch against the main notebook's correct pattern (`terminated, truncated, _ = env.step(...)` then `done = terminated or truncated`), since `done` should really mean "episode over for any reason," not just "terminated." I've used the correct unpacking below to stay consistent with the rest of the notebook, while keeping the adaptive-epsilon logic itself unchanged.


In [ ]:
def adjust_epsilon(score, consecutive_success_threshold=200):
    global epsilon
    if score >= consecutive_success_threshold:
        epsilon *= 0.9      # doing well -> explore less, faster
    else:
        epsilon *= epsilon_decay   # still learning -> decay at the normal rate
    epsilon = max(epsilon_min, epsilon)


In [ ]:
epsilon = 1.0
memory_adaptive = deque(maxlen=2000)
env_adaptive = gym.make('CartPole-v1')

for e in range(episodes):
    state, _ = env_adaptive.reset()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        if np.random.rand() <= epsilon:
            action = random.randrange(action_size)
        else:
            q_values = model.predict(state, verbose=0)
            action = np.argmax(q_values[0])

        # Correct terminated/truncated unpacking (see note above) —
        # the source lab's version conflates `done` with `terminated`.
        next_state, reward, terminated, truncated, _ = env_adaptive.step(action)
        done = terminated or truncated

        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        memory_adaptive.append((state, action, reward, next_state, done))
        state = next_state

        if done:
            adjust_epsilon(time)
            print(f"[Adaptive epsilon] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

env_adaptive.close()


## 🧪 Practice 3 — Custom Reward Function

Rather than the sparse "+1 per timestep, -10 on failure" reward, this shapes a **denser** reward directly from the state — how centered the cart is and how upright the pole is — so the agent gets a graded signal on *every* step instead of only at episode end:

$$r = \Big(1 - \frac{|x|}{2.4}\Big) + \Big(1 - \frac{|\theta|}{0.20948}\Big)$$

Where $x$ is the cart position and $\theta$ is the pole angle. The constants $2.4$ and $0.20948$ rad ($\approx 12°$) are exactly CartPole-v1's own termination thresholds — so both terms are 0 right at the edge of failure and climb toward 1 the closer the agent is to dead-center / perfectly upright. This is the reward-shaping idea from the Theory section made concrete: a continuous quality signal instead of a binary one.

Same unpacking note as Practice 2 applies here (`done`/`terminated` naming) — corrected below for consistency.


In [ ]:
def custom_reward(state):
    x, x_dot, theta, theta_dot = state
    reward = (1 - abs(x) / 2.4) + (1 - abs(theta) / 0.20948)
    return reward


In [ ]:
epsilon = 1.0
memory_custom = deque(maxlen=2000)
env_custom = gym.make('CartPole-v1')

for e in range(episodes):
    state, _ = env_custom.reset()
    state_vec = state  # raw 4-value vector, used directly by custom_reward()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        if np.random.rand() <= epsilon:
            action = random.randrange(action_size)
        else:
            q_values = model.predict(state, verbose=0)
            action = np.argmax(q_values[0])

        next_state, _, terminated, truncated, _ = env_custom.step(action)
        done = terminated or truncated

        reward = custom_reward(next_state) if not done else -10
        next_state_reshaped = np.reshape(next_state, [1, state_size])

        memory_custom.append((state, action, reward, next_state_reshaped, done))
        state = next_state_reshaped

        if epsilon > epsilon_min:
            epsilon *= epsilon_decay

        if done:
            print(f"[Custom reward] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

env_custom.close()


## 📊 Summary

| Concept | What it does | Telecom/RF analogy |
|---|---|---|
| Q-value $Q(s,a)$ | Estimated future reward of an action in a state | Predicted link quality of a channel |
| Bellman target | $r + \gamma \max_{a'}Q(s',a')$ — the regression target for training | Updating a channel estimate from a new pilot measurement |
| Epsilon-greedy | Random action w.p. $\epsilon$, else greedy | Cognitive radio spectrum sensing (explore) vs. locking onto a known channel (exploit) |
| Experience replay | Random minibatch sampling from a transition buffer | Interleaving before FEC decoding — breaks temporal correlation |
| Single network (no target net) | Same network supplies both prediction and target | Adapting a filter against a reference that's also moving — less stable than a fixed reference |
| Reward shaping | Denser, continuous reward instead of sparse terminal-only signal | Continuous quality metric (EVM) vs. binary pass/fail per packet |

**⚠️ Issues flagged in the source lab, preserved but called out:**
- Practice 1's solution redefines `act()` to always return `env.action_space.sample()`, silently discarding the trained model — corrected here to keep genuine epsilon-greedy action selection so the architecture comparison is actually meaningful.
- Practice 1's solution imports `from keras.models import Sequential` (bare `keras`), inconsistent with the notebook's `tensorflow.keras` style elsewhere.
- Practice 2 and 3's solutions unpack `env.step()` as `..., done, truncated, _ = env.step(action)`, binding `done` directly to Gymnasium's `terminated` value rather than combining both — corrected to `terminated, truncated, _ = env.step(...)` then `done = terminated or truncated`, matching the main notebook.
- The evaluation loop calls `env.render()` on an environment created without a `render_mode`, which will not actually display anything in current Gymnasium versions without `render_mode='human'` at creation time.
- No separate target network is used — the same network supplies both the trained prediction and the Bellman target, a known simplification vs. a "real" DQN.


## 🧪 Sandbox

Space for further experiments:
- Add a proper **target network**, synced every N episodes, and compare training stability against the single-network version above.
- Track and plot the score per episode across a longer run (100+ episodes) to see the learning curve.
- Try `CartPole-v1`'s harder cousin, `MountainCar-v0`, where the reward is sparse and exploration matters much more.
- Combine the adaptive epsilon (Practice 2) with the custom reward (Practice 3) in a single run.
